In [ ]:
import pandas as pd
import numpy as np
import os

DIALECTS = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"]
SEEDS = [0, 1, 2, 3, 4]

In [ ]:
prompt_type = "toxic"

df = pd.read_csv(f'./text_level_typo_results/{prompt_type}_prompts_with_multiseed_typos.csv')


common_cols = [
    'category', 
    'standard_prompt',
    'standard_prompt_NSFW_T_prob',
    'standard_prompt_NSFW_T_blocked'
]

DIALECTS = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"]

print("🔄 Splitting by dialect from master file...")

for dialect in DIALECTS:
    dialect_cols = [col for col in df.columns if col.startswith(dialect)]
    
    selected_cols = common_cols + dialect_cols
    
    # Check for missing columns in the master file to prevent errors
    missing_cols = [col for col in selected_cols if col not in df.columns]
    if missing_cols:
        print(f"  ⚠ [Warning] The following columns are missing in the master file during {dialect} extraction: {missing_cols}")
    
    df_split = df[selected_cols]
    
    output_filename = f"./text_level_typo_results/{prompt_type}_results_{dialect}.csv"
    df_split.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print(f"✅ Saved {output_filename} (Columns: {len(selected_cols)})")

print("🎉 All dialect files successfully split!")

In [ ]:
common_cols = [
    'category', 'standard_prompt', 
    'standard_prompt_NSFW_T_prob', 'standard_prompt_NSFW_T_blocked'
]

print("🔄 Splitting dialect files by seed and including scores...")

for dialect in DIALECTS:
    file_path = f"./text_level_typo_results/{prompt_type}_results_{dialect}.csv"
    
    if not os.path.exists(file_path):
        print(f"❌ File {file_path} does not exist! Skipping.")
        continue
        
    df = pd.read_csv(file_path)
    
    dialect_prompt_col = f"{dialect}_prompt"
    dialect_prob_col = f"{dialect}_prompt_NSFW_T_prob"
    dialect_blocked_col = f"{dialect}_prompt_NSFW_T_blocked"
    
    base_cols = common_cols + [dialect_prompt_col, dialect_prob_col, dialect_blocked_col]
    
    for seed in SEEDS:
        typo_col = f"{dialect}_typo_s{seed}"
        target_sim_col = f"{dialect}_target_sim_s{seed}"
        typo_sim_col = f"{dialect}_typo_sim_s{seed}"
        sim_diff_col = f"{dialect}_sim_diff_s{seed}" # 💡 Fix duplicate typo from the original code
        
        typo_prob_col = f"{typo_col}_NSFW_T_prob"
        typo_blocked_col = f"{typo_col}_NSFW_T_blocked"
        
        seed_specific_cols = [
            typo_col, target_sim_col, typo_sim_col, sim_diff_col,
            typo_prob_col, typo_blocked_col
        ]
        
        missing_cols = [col for col in (base_cols + seed_specific_cols) if col not in df.columns]
        if missing_cols:
            print(f"  ⚠ [Warning] {dialect} data is missing seed {seed} related columns: {missing_cols}. Skipping.")
            continue
            
        df_seed = df[base_cols + seed_specific_cols].copy()
        
        # rename columns for 100% compatibility with appendix code
        rename_dict = {
            dialect_prompt_col: 'dialect_prompt',
            dialect_prob_col: 'dialect_prompt_NSFW_T_prob',
            dialect_blocked_col: 'dialect_prompt_NSFW_T_blocked',
            
            typo_col: 'typo_prompt',
            target_sim_col: 'target_sim',
            typo_sim_col: 'typo_sim',
            sim_diff_col: 'sim_difference',
            
            typo_prob_col: 'typo_prompt_NSFW_T_prob',
            typo_blocked_col: 'typo_prompt_NSFW_T_blocked'
        }
        df_seed.rename(columns=rename_dict, inplace=True)
        
        output_filename = f"./text_level_typo_results/{prompt_type}_results_{dialect}_seed_{seed}.csv"
        df_seed.to_csv(output_filename, index=False, encoding='utf-8-sig')
        
        print(f"✅ Saved: {prompt_type}_results_{dialect}_seed_{seed}.csv (Rows: {len(df_seed)})")
        
print("🎉 All seeds split and score columns unified!")

In [ ]:
stats_df = pd.read_csv(f'./text_level_typo_results/raw_{prompt_type}_multiseed_data.csv')  

if prompt_type == 'benign':
    final_df = stats_df.groupby("Dialect").agg(
            Std_FPR_mean=("Std_FPR(%)", "mean"),
            Dialect_FPR_mean=("Dialect_FPR(%)", "mean"),
            Typo_FPR_mean=("Typo_FPR(%)", "mean"),
            Typo_FPR_std=("Typo_FPR(%)", "std"),
            Bias_Gap_mean=("Bias_Gap(Dialect-Typo)", "mean"),
            Bias_Gap_std=("Bias_Gap(Dialect-Typo)", "std")
        ).reset_index().round(3)
elif prompt_type == 'toxic':
    
    stats_df["Bias_Gap(Dialect-Typo)"] = stats_df["Dialect_TPR(%)"] - stats_df["Typo_TPR(%)"]
    
    final_df = stats_df.groupby("Dialect").agg(
            Std_TPR_mean=("Std_TPR(%)", "mean"),
            Dialect_TPR_mean=("Dialect_TPR(%)", "mean"),
            Typo_TPR_mean=("Typo_TPR(%)", "mean"),
            Typo_TPR_std=("Typo_TPR(%)", "std"),
            Bias_Gap_mean=("Bias_Gap(Dialect-Typo)", "mean"),
            Bias_Gap_std=("Bias_Gap(Dialect-Typo)", "std")
        ).reset_index().round(3)

sim_diff_stats = []

print("📊 [typo_sim] Iterating files and extracting statistics...")
for dialect in DIALECTS:
    all_typo_sim = []
    for seed in SEEDS:
        file_path = f"./text_level_typo_results/{prompt_type}_results_{dialect}_seed_{seed}.csv"
        
        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)
            if 'typo_sim' in temp_df.columns:
                all_typo_sim.extend(temp_df['typo_sim'].dropna().tolist())
            else:
                print(f"  ⚠ [Warning] 'typo_sim' column missing in {file_path}!")
        else:
            print(f"  ⚠ [Warning] File {file_path} not found!")

    # Use ddof=1 to apply the same sample standard deviation as pandas
    if all_typo_sim:
        typo_sim_mean = np.mean(all_typo_sim)
        typo_sim_std = np.std(all_typo_sim, ddof=1) 
    else:
        typo_sim_mean, typo_sim_std = None, None

    sim_diff_stats.append({
        'Dialect': dialect,
        'typo_sim_mean': round(typo_sim_mean, 3) if typo_sim_mean is not None else None,
        'typo_sim_std': round(typo_sim_std, 3) if typo_sim_std is not None else None
    })

sim_diff_df = pd.DataFrame(sim_diff_stats)

merged_df = pd.merge(final_df, sim_diff_df, on='Dialect', how='left')

print("\n🔥 Final table completed (typo_sim_mean, typo_sim_std added)!")
print(merged_df)

# Save with a new name to prevent overwriting the original
save_path = f'./text_level_typo_results/FINAL_PAPER_TABLE_Data_{prompt_type}_prompts_with_SimDiff.csv'
merged_df.to_csv(save_path, index=False)
print(f"\n✅ Merged final file saved: {save_path}")

In [ ]:
if prompt_type == "benign": 
    print(merged_df[['Dialect', 'Typo_FPR_mean', 'Typo_FPR_std', 'Bias_Gap_mean', 'Bias_Gap_std']])
else:
    print(merged_df[['Dialect', 'Typo_TPR_mean', 'Typo_TPR_std', 'Bias_Gap_mean', 'Bias_Gap_std']])

In [ ]:
merged_df.keys()

In [ ]:
print(merged_df[['Dialect', 'typo_sim_mean', 'typo_sim_std']])

In [ ]:
typo_sim_stats = []
for dialect in DIALECTS:
    df = pd.read_csv(f"./text_level_typo_results/{prompt_type}_results_{dialect}.csv")
    
    target_cols = [
        f"{dialect}_typo_sim_s0", 
        f"{dialect}_typo_sim_s1", 
        f"{dialect}_typo_sim_s2"
    ]
    
    # Remove NaN values to prevent computation errors
    all_typo_sims = df[target_cols].values.flatten()
    all_typo_sims = all_typo_sims[~np.isnan(all_typo_sims)]
    
    if len(all_typo_sims) > 0:
        typo_sim_mean = np.mean(all_typo_sims)
        typo_sim_std = np.std(all_typo_sims, ddof=1)
    else:
        print(f"  ⚠ [Warning] No valid typo_sim values in {dialect} data!")
        typo_sim_mean, typo_sim_std = None, None
  

    typo_sim_stats.append({
        'Dialect': dialect,
        'typo_sim_mean': round(typo_sim_mean, 6) if typo_sim_mean is not None else None,
        'typo_sim_std': round(typo_sim_std, 6) if typo_sim_std is not None else None
    })

typo_sim_df = pd.DataFrame(typo_sim_stats)